In [ ]:
import shutil
from pathlib import Path

import git

In [ ]:
def download_skill_from_repo(
    skill_name: str,
    repo_dir: str | Path,
    output_dir: str | Path,
) -> bool:
    """
    Recursively finds a directory matching skill_name within repo_dir
    and clones it to the output directory.
    """
    root_source = Path(repo_dir)
    target_path = Path(output_dir) / skill_name

    # 1. Validation: Check if root template directory exists
    if not root_source.is_dir():
        print(f"[Error] Root template directory not found: {root_source}")
        return False

    # 2. Validation: Prevent overwriting existing target
    if target_path.exists():
        print(f"[Skip] Target directory already exists: {target_path}")
        return False

    # 3. Discovery: Find the skill directory anywhere under the template root
    # rglob("*") searches recursively; we check for is_dir() and name match
    source_skill_path = next(
        (p for p in root_source.rglob(f"*/{skill_name}") if p.is_dir()), None
    )

    if not source_skill_path:
        # Fallback: check if the root itself is the skill (unlikely but safe)
        if root_source.name == skill_name:
            source_skill_path = root_source
        else:
            print(
                f"[Error] Skill folder '{skill_name}' not found within: {root_source}"
            )
            return False

    try:
        # 4. Execution: Clone the discovered directory
        shutil.copytree(source_skill_path, target_path)
        print(f"[Success] Found '{skill_name}' at {source_skill_path}")
        print(f"[Success] Initialized to: {target_path}")
        return True

    except Exception as e:
        print(f"[Failure] Error during initialization: {e}")
        return False

In [ ]:
skill_source_dir = Path("./skill_projects")
output_dir = Path("../.github/skills")

In [ ]:
skill_dict = {
    # "repo-name": {"skill1", "skill2", ...}
    "anthropics/skills": {
        "xlsx",
        "pptx",
        "docx",
        "pdf",
        "theme-factory",
        "skill-creator",
        "doc-coauthoring",
    },
    "NeoLabHQ/context-engineering-kit": {
        "agent-evaluation",
        "apply-anthropic-skill-best-practices",
        "context-engineering",
        "create-command",
        "create-hook",
        "create-skill",
        "prompt-engineering",
        "thought-based-reasoning",
        "software-architecture",
    },
}

In [ ]:
for project, skills in skill_dict.items():
    project_name = project.split("/")[-1]
    repo_dir = skill_source_dir / project_name
    if not repo_dir.exists():
        # Clone the project if it doesn't exist locally
        print(f"Cloning project '{project}' into '{repo_dir}'...")
        repo_url = f"https://github.com/{project}.git"
        git.Repo.clone_from(repo_url, repo_dir)
        print(f"Cloned '{project}' successfully.")
    for skill in skills:
        success = download_skill_from_repo(
            skill_name=skill,
            repo_dir=repo_dir,
            output_dir=output_dir,
        )
        if not success:
            print(f"[Warning] Failed to process skill: {skill}")
    print("=" * 50)